# Layerwise stream RankMe over training — baseline vs layer-ablated

Ablated twin of experiments_anim_4: each animation shows the depth profile of stream
RankMe (centered, `attn.in` per block + `before_final_norm` + `after_final_norm`) with
the unablated sweep on the left and the layer-ablated run (writes zeroed) on the right.
Both panels are restricted to the runs' common checkpoints and share one y-range.
For now: pythia-1b (− blk3, the carrier) and nanochat-d12 (− blk3-5).

In [ ]:
import os, sys
import shutil
import warnings
import importlib
if os.path.basename(os.getcwd()) == 'analysis':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
import numpy as np
import utils.model_registry, utils.accessor
importlib.reload(utils.model_registry)   # deps first: reload(_lib) alone re-imports cached modules
importlib.reload(utils.accessor)
from analysis import experiments_lib as _lib
importlib.reload(_lib)
from analysis.experiments_lib import (
    get_ys, get_series_y, panel_palettes, smooth_spectrum,
    submatrix, block_mean_cos, model_name_options, YVAR_LABELS, XVAR_FNS)
warnings.filterwarnings('ignore', message='Data has no positive values')

In [ ]:
# model -> (baseline cfg, ablated cfg); both panels are cut to the checkpoints both runs share.
PAIR = {'pythia-1b-deduped': ('block_representations_samples', 'ablate_blk3'),
        'nanochat-d12': ('nanochat_samples', 'ablate_blk3-5')}
measured_br = {'pythia-1b-deduped': list(range(16)), 'nanochat-d12': list(range(12))}

COMMON = {}                                  # (cfg, model) -> the pair's shared step set
for model, (base, abl) in PAIR.items():
    steps = [set(np.load(f'data/results/{c}/results_{model}.npy', allow_pickle=True).item())
             for c in (base, abl)]
    COMMON[(base, model)] = COMMON[(abl, model)] = steps[0] & steps[1]

def get_ys_aligned(cfg, model, hook, yvar, *a, **kw):
    ys, steps = get_ys(cfg, model, hook, yvar, *a, **kw)
    allowed = COMMON.get((cfg, model))
    if ys is None or allowed is None:
        return ys, steps
    idx = [i for i, s in enumerate(steps) if s in allowed]
    return [ys[i] for i in idx], [steps[i] for i in idx]

In [ ]:
import analysis.spectrum_anim as sa
importlib.reload(sa)   # pick up engine edits without restarting the kernel
sa.configure(get_ys=get_ys_aligned, get_series_y=get_series_y,
             panel_palettes=panel_palettes, smooth_spectrum=smooth_spectrum,
             submatrix=submatrix, block_mean_cos=block_mean_cos,
             model_name_options=model_name_options, YVAR_LABELS=YVAR_LABELS, XVAR_FNS=XVAR_FNS)

In [ ]:
from IPython.display import display

def anim_layer_rankme_ablated(model, save_dir=None, fps=4):
    base, abl = PAIR[model]
    def layers(src):
        ls = [(src, (f'blk{k}.attn.in', 'acts_centered'), f'blk{k}') for k in measured_br[model]]
        return ls + [(src, ('before_final_norm', 'acts_centered'), 'bfn'),
                     (src, ('after_final_norm', 'acts_centered'), 'afn')]
    tag = abl[len('ablate_'):]
    panels = [('rankme', layers(base), [model],
               {'kind': 'profile', 'ylog': True, 'title': 'baseline'}),
              ('rankme', layers(abl), [model],
               {'kind': 'profile', 'ylog': True, 'title': f'− {tag}'})]
    # materialize directly (mirrors animate_spectra) so the two panels can share one y-range
    spec = sa._materialize(panels, 2, fps, model, 'tokens', 'bottom',
                           f'Layerwise stream RankMe — {model}, baseline vs − {tag}', None)
    lims = [p['ylim'] for p in spec['panels']]
    shared = (min(l[0] for l in lims), max(l[1] for l in lims))
    for p in spec['panels']:
        p['ylim'] = shared
    if save_dir:
        save = f'{save_dir}/layer_rankme_ablated_{model}.mp4'
        if shutil.which('ffmpeg') is None:
            sa._save_mp4_via_sbatch(spec, save)
        else:
            sa.render(spec, save)
    display(sa.render(spec, None))

In [ ]:
anim_layer_rankme_ablated('pythia-1b-deduped', save_dir='analysis/figures/animations')

In [ ]:
anim_layer_rankme_ablated('nanochat-d12', save_dir='analysis/figures/animations')